In [ ]:
import numpy as np
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# custom
from _config import get_prms
from _trainer import Trainer
# devisce
device = torch.device("cuda:9" if torch.cuda.is_available() else "cpu")
print(device)

seed = 1234
np.random.seed(seed)
torch.manual_seed(seed)

# problem settings : exponential, logistic, target cell-limited 
name = 'log' # 'exp', 'log', 'lorenz'
meth_i = 2
dist_i = 3
methods = ['gp', 'deeponet', 'eidgm']
dist_list = ['uni', 'bi', 'tri']
dist_type = dist_list[dist_i-1] # 'uni', 'bi', 'tri'
get_plot = True

# hyperparameter settings (prms[0]:DE, prms[1]:hyperPINN, prms[2]:WGAN)
prms = get_prms(name, dist_type=dist_type) # DE / hyperPINN / WGAN settings
prms[1]['deeponet'] = (meth_i==2)
if get_plot:
    prms[2]['num_noised'] = 1000//dist_i # Insert the number of real paramaters from the each nodes with noise 
    prms[2]['num_gen_plot'] = (1000-1)*prms[2]['num_bins']
else:
    prms[2]['num_noised'] = 1000//dist_i # Insert the number of real paramaters from the each nodes with noise 
    prms[2]['num_gen_plot'] = (1000-1)*prms[2]['num_bins']
#prms[2]['num_gen_ratio'] = 1 # Insert the ratio : fake data/real data 
prms[2]['num_cut'] = 0 # Insert the number of censored time for each mode which is randomly determined via numpy seed.
#prms

# get dataset : if you want real data 
real_data = None # None, 'Abeta40', 'Abeta42'
trainer = Trainer(name, prms, device=device)
modes_noisy, X_data, X_data_onehot = trainer.get_data_gan(real_data=real_data)

> model output : gp / deeponet / hyperpinn 

In [ ]:
if meth_i in [2,3]:
    # prepare models
    # load pre-trained hyperPINNs
    trainer.load_pinn()

    # load WGANs
    load_params = True # if you want to train WGAN from the begining, then this should be False.
    trainer.load_gan(load_params=load_params)

    # show WGAN result
    gens = trainer.plot_results_gan(modes_noisy, X_data, plot_traj=False, return_gen=True, get_scores=True)

    print(len(modes_noisy), len(gens))
else:
    # GET GP save files
    gens = torch.load('./save/GP/gp_'+trainer.name+str(trainer.num_modes)+'.pth')
    print(len(modes_noisy), len(gens), trainer.WD(gens, modes_noisy))

> plot

In [ ]:
trues = pd.DataFrame(np.concatenate([modes_noisy.numpy(), np.reshape(np.array([['true']*len(modes_noisy)]),(-1,1))],-1), columns=trainer.param_names+[' ']).astype({key:float for key in trainer.param_names})
fakes = pd.DataFrame(np.concatenate([gens.numpy(), np.reshape(np.array([['estimated']*len(gens)]),(-1,1))],-1), columns=trainer.param_names+[' ']).astype({key:float for key in trainer.param_names})
df = pd.concat([fakes, trues])

cc = ['magenta','blue','green']
tc = cc[dist_i-1]
fs = 27
sns.set(font_scale=3)
sns.set_style("ticks", {'axes.grid' : False})

c_mod = tc
num_bins = trainer.num_bins
tmin, tmax = trainer.tmin, trainer.tmax
x_bins = trainer.t_numpy
t = np.linspace(tmin,tmax,num_bins)
t_torch = torch.linspace(tmin,tmax,num_bins)
# Solve and plot trajectories
y_values = []
for mod in modes_noisy:
    y_values.append(trainer.solution(torch.cat([t_torch.view(-1,1), mod.view(-1,trainer.num_p).repeat(num_bins,1)], -1)))
    
if dist_i==1:
    fs = 32
    xlb1=False
    ylb1=False
    bw = 5
if dist_i==2:
    fs = 32
    xlb1=False
    ylb1=True
    bw = 0.1
if dist_i==3:
    fs = 35
    xlb1=True
    ylb1=False
    bw = 0.1

In [ ]:
if meth_i == 1:
    # true data plot 
    fig, axs = plt.subplots(1, 1, figsize=(6+1*int(ylb1)+0.3*int(xlb1), 5+0.9*int(xlb1)), dpi=300)

    # true data plot 
    k = 0
    for yval in y_values:
        plt.scatter(t_torch, yval[:,k], color=tc, s=40, alpha=1)

    # Plot formatting
    axs.set_xlim([-1, 21])
    axs.set_ylim([-0.1, 1.5])
    axs.set_xticks([0,10,20])
    axs.set_yticks([0, 0.7, 1.4])
    if xlb1:
        axs.set_xlabel('Time', fontsize=fs)  # Larger font size for formal papers
    else:
        axs.set_xlabel('', fontsize=fs)  # Larger font size for formal papers
        axs.set_xticklabels([])
    if ylb1:
        axs.set_ylabel('Population', fontsize=fs)  # Larger font size for formal papers
    else:
        axs.set_ylabel('', fontsize=fs)  # Larger font size for formal papers
        axs.set_yticklabels([])
    axs.tick_params(axis='both', which='major', labelsize=24)
    #plt.title('Observation', fontsize=16, fontweight='bold')  # Title for the plot
    # Use a tight layout to ensure everything fits without overlap
    for axis in ['bottom','left']:
        axs.spines[axis].set_linewidth(4)
    for axis in ['top','right']:
        axs.spines[axis].set_linewidth(0)
    axs.yaxis.set_tick_params(width=3)
    axs.xaxis.set_tick_params(width=3)
    plt.tight_layout()

    # Use a tight layout to ensure everything fits without overlap
    plt.tight_layout()
    # Save the figure with high resolution for better clarity in the paper
    plt.savefig('./eps/log'+str(dist_i)+'_traj.eps', format='eps', dpi=300)
    # Show the plot
    plt.show()

In [ ]:
if meth_i == 1:
    plt.figure(figsize=(4,4), dpi=300)
    g = sns.distplot(df.loc[df[' '] == 'true', 'r'], hist=False, rug=False, color=tc, kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
    if xlb1:
        g.set(ylabel='', yticks=[], xticks=[1.0,2.5,4.0], xlim=[-0.5, 5.5])
    else:
        g.set(ylabel='', xlabel='', yticks=[], xticks=[1.0,2.5,4.0], xticklabels=[], xlim=[-0.5, 5.5])
    for axis in ['bottom','left']:
        g.spines[axis].set_linewidth(4)
    for axis in ['top','right']:
        g.spines[axis].set_linewidth(0)
    g.xaxis.set_tick_params(width=3)
    plt.savefig('./eps/log'+str(dist_i)+'_true_r.eps', format='eps', dpi=300)
    plt.show()

In [ ]:
plt.figure(figsize=(4,4), dpi=300)
g = sns.distplot(df.loc[df[' '] == 'true', 'r'], hist=False, rug=False, color=tc, kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
g = sns.distplot(df.loc[df[' '] == 'estimated', 'r'], hist=False, rug=False, color='orange', kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
g.set(ylabel='', xlabel='', yticks=[], xticks=[1.0,2.5,4.0], xticklabels=[], xlim=[-0.5, 5.5])
for axis in ['bottom','left']:
    g.spines[axis].set_linewidth(4)
for axis in ['top','right']:
    g.spines[axis].set_linewidth(0)
g.xaxis.set_tick_params(width=3)
plt.savefig('./eps/log'+str(dist_i)+'_'+methods[meth_i-1]+'_fake_r.eps', format='eps', dpi=300)
plt.show()

In [ ]:
if meth_i == 1:
    plt.figure(figsize=(4,4), dpi=300)
    g = sns.distplot(df.loc[df[' '] == 'true', 'K'], hist=False, rug=False, color=tc, kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
    if xlb1:
        g.set(ylabel='', yticks=[], xticks=[0.6,1.0,1.4], xlim=[0.3, 1.7])
    else:
        g.set(ylabel='', xlabel='', yticks=[], xticks=[0.6,1.0,1.4], xticklabels=[], xlim=[0.3, 1.7])
    for axis in ['bottom','left']:
        g.spines[axis].set_linewidth(4)
    for axis in ['top','right']:
        g.spines[axis].set_linewidth(0)
    g.xaxis.set_tick_params(width=3)
    plt.savefig('./eps/log'+str(dist_i)+'_true_K.eps', format='eps', dpi=300)
    plt.show()

In [ ]:
plt.figure(figsize=(4,4), dpi=300)
g = sns.distplot(df.loc[df[' '] == 'true', 'K'], hist=False, rug=False, color=tc, kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
g = sns.distplot(df.loc[df[' '] == 'estimated', 'K'], hist=False, rug=False, color='orange', kde_kws={'linewidth':5, 'fill':True, 'bw':bw})
g.set(ylabel='', xlabel='', yticks=[], xticks=[0.6,1.0,1.4], xticklabels=[], xlim=[0.3, 1.7])
for axis in ['bottom','left']:
    g.spines[axis].set_linewidth(4)
for axis in ['top','right']:
    g.spines[axis].set_linewidth(0)
g.xaxis.set_tick_params(width=3)
plt.savefig('./eps/log'+str(dist_i)+'_'+methods[meth_i-1]+'_fake_K.eps', format='eps', dpi=300)
plt.show()